In [1]:
# Error Handling Starter Code
class FraudDetectionError(Exception):
    pass

class DataValidationError(FraudDetectionError):
    pass

print("🛡️ WEEK 4: Production Error Handling")
print("=" * 50)
print("Building robust error handling system...")

🛡️ WEEK 4: Production Error Handling
Building robust error handling system...


In [4]:
# WEEK 4: Production Error Handling - FIXED VERSION
# =================================================

print("🛡️ WEEK 4: Production Error Handling")
print("=" * 50)
print("🎯 Goals: Robust error handling, input validation, graceful degradation")

import pandas as pd
import numpy as np
import joblib
import logging
import sys
from datetime import datetime
import traceback

# =============================================================================
# FIX 1: PROPER LOGGING SETUP (No Unicode Issues)
# =============================================================================

# Set up logging with ASCII-only characters to avoid Unicode issues
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(levelname)s - %(message)s',
    handlers=[
        logging.FileHandler('fraud_detection.log', encoding='utf-8'),
        logging.StreamHandler(sys.stdout)
    ]
)

logger = logging.getLogger(__name__)

# ASCII-only success indicators
SUCCESS = "[SUCCESS]"
FAILED = "[FAILED]"
VALIDATION_PASSED = "[VALIDATION_PASSED]"

print("✅ Logging system initialized (ASCII-only for compatibility)")

# =============================================================================
# 1. CUSTOM EXCEPTION CLASSES
# =============================================================================

print("\n🚨 1. CUSTOM EXCEPTION CLASSES")
print("=" * 40)

class FraudDetectionError(Exception):
    """Base exception for fraud detection system"""
    pass

class DataValidationError(FraudDetectionError):
    """Raised when input data fails validation"""
    pass

class ModelPredictionError(FraudDetectionError):
    """Raised when model prediction fails"""
    pass

class FeatureEngineeringError(FraudDetectionError):
    """Raised when feature engineering fails"""
    pass

class ConfigurationError(FraudDetectionError):
    """Raised when system configuration is invalid"""
    pass

print("✅ Custom exception classes defined")

# =============================================================================
# 2. INPUT VALIDATION SYSTEM
# =============================================================================

print("\n🔍 2. INPUT VALIDATION SYSTEM")
print("=" * 40)

class ClaimDataValidator:
    """Validates incoming claim data for fraud detection"""
    
    REQUIRED_FIELDS = ['patient_age', 'claimed_amount', 'billed_items_count',
                      'previous_claims_count', 'doc_missing_flag', 'hospital_id', 'insurer_id']
    
    FIELD_RANGES = {
        'patient_age': (0, 120),
        'claimed_amount': (0, 10_000_000),  # $10M max
        'billed_items_count': (0, 1000),
        'previous_claims_count': (0, 1000),
        'doc_missing_flag': (0, 1),
        'hospital_id': (1, 10000),
        'insurer_id': (1, 1000)
    }
    
    @classmethod
    def validate_single_claim(cls, claim_data):
        """Validate a single claim record"""
        logger.info("Validating single claim data")
        
        # Check required fields
        missing_fields = [field for field in cls.REQUIRED_FIELDS if field not in claim_data]
        if missing_fields:
            raise DataValidationError(f"Missing required fields: {missing_fields}")
        
        # Validate field types and ranges
        for field, value in claim_data.items():
            if field in cls.FIELD_RANGES:
                try:
                    numeric_value = float(value)
                    min_val, max_val = cls.FIELD_RANGES[field]
                    if not (min_val <= numeric_value <= max_val):
                        raise DataValidationError(
                            f"Field {field} value {value} outside valid range [{min_val}, {max_val}]"
                        )
                except (ValueError, TypeError):
                    raise DataValidationError(f"Field {field} must be numeric, got {value}")
        
        logger.info(f"{VALIDATION_PASSED} Single claim validation passed")
        return True
    
    @classmethod
    def validate_batch_claims(cls, batch_data):
        """Validate a batch of claims"""
        logger.info(f"Validating batch of {len(batch_data)} claims")
        
        if not isinstance(batch_data, (list, pd.DataFrame)):
            raise DataValidationError("Batch data must be list or DataFrame")
        
        if len(batch_data) == 0:
            raise DataValidationError("Batch data cannot be empty")
        
        if len(batch_data) > 10000:  # Reasonable batch limit
            raise DataValidationError(f"Batch size {len(batch_data)} exceeds maximum 10,000")
        
        # Validate each claim in batch
        for i, claim in enumerate(batch_data):
            try:
                cls.validate_single_claim(claim)
            except DataValidationError as e:
                raise DataValidationError(f"Claim at index {i} invalid: {str(e)}")
        
        logger.info(f"{VALIDATION_PASSED} Batch validation passed")
        return True

print("✅ Input validation system created")

# =============================================================================
# 3. ERROR HANDLING DECORATORS
# =============================================================================

print("\n🎯 3. ERROR HANDLING DECORATORS")
print("=" * 40)

def handle_prediction_errors(func):
    """Decorator to handle prediction errors gracefully"""
    def wrapper(*args, **kwargs):
        try:
            return func(*args, **kwargs)
        except DataValidationError as e:
            logger.error(f"Data validation error: {str(e)}")
            return {
                'success': False,
                'error': f'Invalid input data: {str(e)}',
                'error_type': 'VALIDATION_ERROR'
            }
        except ModelPredictionError as e:
            logger.error(f"Model prediction error: {str(e)}")
            return {
                'success': False, 
                'error': f'Prediction failed: {str(e)}',
                'error_type': 'PREDICTION_ERROR'
            }
        except Exception as e:
            logger.error(f"Unexpected error: {str(e)}\n{traceback.format_exc()}")
            return {
                'success': False,
                'error': 'Internal server error',
                'error_type': 'INTERNAL_ERROR'
            }
    return wrapper

def log_execution_time(func):
    """Decorator to log function execution time"""
    def wrapper(*args, **kwargs):
        start_time = datetime.now()
        result = func(*args, **kwargs)
        execution_time = (datetime.now() - start_time).total_seconds()
        
        logger.info(f"Function {func.__name__} executed in {execution_time:.4f} seconds")
        
        if isinstance(result, dict) and 'success' in result:
            result['execution_time_seconds'] = execution_time
            
        return result
    return wrapper

print("✅ Error handling decorators defined")

# =============================================================================
# 4. ROBUST PREDICTION PIPELINE (WITH SKLEARN WARNING FIX)
# =============================================================================

print("\n🔧 4. ROBUST PREDICTION PIPELINE")
print("=" * 40)

class RobustFraudPredictor:
    """Production-ready fraud prediction system with comprehensive error handling"""
    
    def __init__(self, model_path='fraud_detection_model.pkl', feature_columns_path='feature_columns.pkl'):
        try:
            self.model = joblib.load(model_path)
            self.feature_columns = joblib.load(feature_columns_path)
            self.validator = ClaimDataValidator()
            logger.info(f"{SUCCESS} Fraud predictor initialized successfully")
        except Exception as e:
            raise ConfigurationError(f"Failed to initialize predictor: {str(e)}")
    
    @handle_prediction_errors
    @log_execution_time
    def predict_single(self, claim_data):
        """Predict fraud for a single claim with full error handling"""
        logger.info("Processing single claim prediction")
        
        # Validate input
        self.validator.validate_single_claim(claim_data)
        
        # Prepare features
        try:
            features = self._prepare_features(claim_data)
        except Exception as e:
            raise FeatureEngineeringError(f"Feature preparation failed: {str(e)}")
        
        # Make prediction (with sklearn warning suppression)
        try:
            import warnings
            from sklearn.exceptions import DataConversionWarning
            
            with warnings.catch_warnings():
                warnings.filterwarnings("ignore", category=DataConversionWarning)
                warnings.filterwarnings("ignore", category=UserWarning)
                
                prediction = self.model.predict(features)[0]
                probability = self.model.predict_proba(features)[0, 1]
                
        except Exception as e:
            raise ModelPredictionError(f"Model prediction failed: {str(e)}")
        
        result = {
            'success': True,
            'prediction': int(prediction),
            'probability': float(probability),
            'risk_level': self._get_risk_level(probability),
            'timestamp': datetime.now().isoformat()
        }
        
        logger.info(f"{SUCCESS} Single prediction completed: {result}")
        return result
    
    @handle_prediction_errors  
    @log_execution_time
    def predict_batch(self, batch_data):
        """Predict fraud for a batch of claims"""
        logger.info(f"Processing batch prediction for {len(batch_data)} claims")
        
        # Validate batch
        self.validator.validate_batch_claims(batch_data)
        
        # Prepare all features
        try:
            features_list = [self._prepare_features(claim) for claim in batch_data]
            features_array = np.vstack(features_list)
        except Exception as e:
            raise FeatureEngineeringError(f"Batch feature preparation failed: {str(e)}")
        
        # Make batch predictions (with sklearn warning suppression)
        try:
            import warnings
            from sklearn.exceptions import DataConversionWarning
            
            with warnings.catch_warnings():
                warnings.filterwarnings("ignore", category=DataConversionWarning)
                warnings.filterwarnings("ignore", category=UserWarning)
                
                predictions = self.model.predict(features_array)
                probabilities = self.model.predict_proba(features_array)[:, 1]
                
        except Exception as e:
            raise ModelPredictionError(f"Batch prediction failed: {str(e)}")
        
        results = []
        for i, (pred, prob) in enumerate(zip(predictions, probabilities)):
            results.append({
                'claim_index': i,
                'prediction': int(pred),
                'probability': float(prob),
                'risk_level': self._get_risk_level(prob)
            })
        
        batch_result = {
            'success': True,
            'predictions': results,
            'batch_size': len(batch_data),
            'fraud_count': int(predictions.sum()),
            'timestamp': datetime.now().isoformat()
        }
        
        logger.info(f"{SUCCESS} Batch prediction completed: {batch_result['fraud_count']} fraud cases detected")
        return batch_result
    
    def _prepare_features(self, claim_data):
        """Prepare features for model prediction with proper feature names"""
        try:
            # Create feature array in correct order with proper feature names
            features = []
            for col in self.feature_columns:
                features.append(float(claim_data[col]))
            
            # Convert to DataFrame with proper feature names to avoid sklearn warnings
            features_df = pd.DataFrame([features], columns=self.feature_columns)
            return features_df
            
        except Exception as e:
            raise FeatureEngineeringError(f"Failed to prepare features: {str(e)}")
    
    def _get_risk_level(self, probability):
        """Convert probability to risk level"""
        if probability >= 0.7:
            return "HIGH"
        elif probability >= 0.3:
            return "MEDIUM" 
        else:
            return "LOW"

print("✅ Robust prediction pipeline created (with sklearn warning fixes)")

# =============================================================================
# 5. TEST THE ERROR HANDLING SYSTEM
# =============================================================================

print("\n🧪 5. TESTING ERROR HANDLING SYSTEM")
print("=" * 40)

# Initialize the robust predictor
try:
    predictor = RobustFraudPredictor()
    print(f"{SUCCESS} Predictor initialized successfully")
except Exception as e:
    print(f"{FAILED} Predictor initialization failed: {e}")

# Test 1: Valid single prediction
print("\n🔸 TEST 1: Valid single prediction")
valid_claim = {
    'patient_age': 45,
    'claimed_amount': 5000.0,
    'billed_items_count': 12,
    'previous_claims_count': 3, 
    'doc_missing_flag': 0,
    'hospital_id': 101,
    'insurer_id': 5
}

result = predictor.predict_single(valid_claim)
print(f"Result: {result}")

# Test 2: Invalid data (missing field)
print("\n🔸 TEST 2: Invalid data (missing field)")
invalid_claim = {
    'patient_age': 45,
    'claimed_amount': 5000.0,
    # Missing required fields
}

result = predictor.predict_single(invalid_claim)
print(f"Result: {result}")

# Test 3: Invalid data (out of range)
print("\n🔸 TEST 3: Invalid data (out of range)")
out_of_range_claim = {
    'patient_age': 150,  # Too high
    'claimed_amount': 5000.0,
    'billed_items_count': 12,
    'previous_claims_count': 3,
    'doc_missing_flag': 0, 
    'hospital_id': 101,
    'insurer_id': 5
}

result = predictor.predict_single(out_of_range_claim)
print(f"Result: {result}")

# Test 4: Valid batch prediction
print("\n🔸 TEST 4: Valid batch prediction")
batch_claims = [valid_claim] * 3  # 3 identical claims
result = predictor.predict_batch(batch_claims)
print(f"Batch result - Success: {result['success']}, Fraud count: {result['fraud_count']}")

# =============================================================================
# 6. GRACEFUL DEGRADATION - FALLBACK SYSTEM
# =============================================================================

print("\n🔄 6. GRACEFUL DEGRADATION SYSTEM")
print("=" * 40)

class FallbackFraudDetector:
    """Simple rule-based fallback when ML model fails"""
    
    @staticmethod
    def rule_based_predict(claim_data):
        """Rule-based fraud detection as fallback"""
        try:
            risk_score = 0
            
            # Rule 1: High claim amount
            if claim_data['claimed_amount'] > 10000:
                risk_score += 2
            
            # Rule 2: Many previous claims
            if claim_data['previous_claims_count'] > 5:
                risk_score += 1
            
            # Rule 3: Missing documentation
            if claim_data['doc_missing_flag'] == 1:
                risk_score += 2
            
            # Rule 4: Young patient with high claims
            if claim_data['patient_age'] < 30 and claim_data['claimed_amount'] > 5000:
                risk_score += 1
            
            # Convert to probability-like score
            probability = min(risk_score / 6.0, 1.0)
            prediction = 1 if probability > 0.5 else 0
            
            return {
                'success': True,
                'prediction': prediction,
                'probability': probability,
                'risk_level': 'HIGH' if probability > 0.7 else 'MEDIUM' if probability > 0.3 else 'LOW',
                'fallback_used': True,
                'timestamp': datetime.now().isoformat()
            }
        except Exception as e:
            return {
                'success': False,
                'error': f'Fallback system failed: {str(e)}',
                'fallback_used': True
            }

# Enhanced predictor with fallback
class ResilientFraudPredictor(RobustFraudPredictor):
    """Predictor with automatic fallback to rule-based system"""
    
    @handle_prediction_errors
    @log_execution_time
    def predict_single_resilient(self, claim_data):
        """Predict with automatic fallback on failure"""
        try:
            # Try ML prediction first
            ml_result = self.predict_single(claim_data)
            if ml_result['success']:
                ml_result['fallback_used'] = False
                return ml_result
        except Exception:
            pass  # Fall through to fallback
        
        # Use fallback system
        logger.warning("ML prediction failed, using rule-based fallback")
        fallback_result = FallbackFraudDetector.rule_based_predict(claim_data)
        fallback_result['fallback_used'] = True
        return fallback_result

print("✅ Graceful degradation system implemented")

# Test the resilient predictor
print("\n🔸 TEST 5: Resilient predictor with fallback")
resilient_predictor = ResilientFraudPredictor()

# This should use ML model
result = resilient_predictor.predict_single_resilient(valid_claim)
print(f"Resilient result: {result}")

# =============================================================================
# 7. SAVE ERROR HANDLING CONFIGURATION
# =============================================================================

print("\n💾 7. SAVING ERROR HANDLING CONFIGURATION")
print("=" * 40)

error_handling_config = {
    'validator_rules': ClaimDataValidator.FIELD_RANGES,
    'required_fields': ClaimDataValidator.REQUIRED_FIELDS,
    'batch_size_limit': 10000,
    'fallback_enabled': True,
    'log_level': 'INFO',
    'config_timestamp': datetime.now().isoformat()
}

joblib.dump(error_handling_config, 'error_handling_config.pkl')

print("✅ Error handling configuration saved")
print(f"{SUCCESS} Production error handling system completed!")

print(f"\n🎉 READY FOR NEXT NOTEBOOK: 09_demo_preparation.ipynb")

🛡️ WEEK 4: Production Error Handling
🎯 Goals: Robust error handling, input validation, graceful degradation
✅ Logging system initialized (ASCII-only for compatibility)

🚨 1. CUSTOM EXCEPTION CLASSES
✅ Custom exception classes defined

🔍 2. INPUT VALIDATION SYSTEM
✅ Input validation system created

🎯 3. ERROR HANDLING DECORATORS
✅ Error handling decorators defined

🔧 4. ROBUST PREDICTION PIPELINE
✅ Robust prediction pipeline created (with sklearn warning fixes)

🧪 5. TESTING ERROR HANDLING SYSTEM
2025-11-15 13:26:20,610 - INFO - [SUCCESS] Fraud predictor initialized successfully
[SUCCESS] Predictor initialized successfully

🔸 TEST 1: Valid single prediction
2025-11-15 13:26:20,614 - INFO - Processing single claim prediction
2025-11-15 13:26:20,617 - INFO - Validating single claim data
2025-11-15 13:26:20,619 - INFO - [VALIDATION_PASSED] Single claim validation passed
2025-11-15 13:26:20,653 - INFO - [SUCCESS] Single prediction completed: {'success': True, 'prediction': 0, 'probability':